# 02 Wrangling: from long time series to one row per stay

**Input:** the raw record files (one long table of Time, Parameter, Value per stay).
**Output:** `data/processed/features.parquet`, one row per `RecordID` with summary features, the label, and the split/fold from notebook 01.

## Steps
1. Read all 12,000 records into one long table (cached as `data/processed/long.parquet`)
2. Separate the static descriptors (Age, Gender, Height, ICUType, admission Weight)
3. Clean: keep the first 48 h, remove physically impossible values (rule table in `src/features.py`)
4. Build features per stay: min, max, mean, last, count and a `has_<var>` flag for each variable
5. Join the label and the split, save
6. Sanity checks, and a first look at whether missingness carries information

## Principle: nothing here learns from the data
Every step works on one stay at a time with fixed rules. Imputation and scaling *do* learn from the data (a mean, a standard deviation), so they wait for the modeling notebook, where they are fitted on the training folds only.

In [1]:
# ── Cell 1: setup and paths ─────────────────────────────────────────
# INSIGHTS
# - Same path logic as notebook 01, so it runs from notebooks/ on any machine.
# - Result: runs with pandas 3.1.1 on the icu-mortality kernel.
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_io import SETS, read_set, read_outcomes
from src import features as F

LABEL = "In-hospital_death"
pd.set_option("display.max_columns", 50)
print(f"pandas {pd.__version__}, repo root {ROOT}")

pandas 3.1.1, repo root C:\Users\user\source\icu-mortality


In [2]:
# ── Cell 2: read every record into one long table ───────────────────
# INSIGHTS
# - read_set opens each archive once and streams the files. Opening set-c.tar.gz
#   once per record would decompress it 4,000 times.
# - The raw long table is cached as parquet, so later runs take about a second.
# - Result: 5,285,818 rows, 12,000 stays, 42 parameter names (36 measured variables,
#   MechVent, 5 descriptors). Read from the archives in about 21 s, from the cache in 0.1 s.
#   Latest time stamp is exactly 2,880 min (48 h), so no record runs past the window.
LONG_PATH = PROCESSED / "long.parquet"
t0 = time.time()
if LONG_PATH.exists():
    long = pd.read_parquet(LONG_PATH)
    source = "cache"
else:
    long = pd.concat([read_set(RAW, s) for s in SETS], ignore_index=True)
    long.to_parquet(LONG_PATH, index=False)
    source = "raw archives"
print(f"Loaded from {source} in {time.time() - t0:.1f} s")
print(f"Rows: {len(long):,}   stays: {long['RecordID'].nunique():,}   "
      f"distinct parameters: {long['Parameter'].nunique()}")
print(f"Latest time stamp: {long['minutes'].max()} min ({long['minutes'].max() / 60:.0f} h)")
display(long.head())
display(long.groupby("set").agg(rows=("Value", "size"), stays=("RecordID", "nunique")))

Loaded from cache in 0.1 s


Rows: 5,285,818   stays: 12,000   distinct parameters: 42
Latest time stamp: 2880 min (48 h)


,RecordID,set,minutes,Parameter,Value
0,141834,a,0,RecordID,141834.0
1,141834,a,0,Age,52.0
2,141834,a,0,Gender,1.0
3,141834,a,0,Height,172.7
4,141834,a,0,ICUType,2.0


,rows,stays
set,,
a,1757980,4000
b,1762535,4000
c,1765303,4000


In [3]:
# ── Cell 3: static descriptors ──────────────────────────────────────
# INSIGHTS
# - Age, Gender, Height, ICUType and Weight are written once at minute 0.
# - Real minute-0 measurements (HR, GCS...) are mixed in between them, so we take
#   the first row of each descriptor at minute 0, not the first 6 rows of the file.
# - Weight is also measured later; only its first minute-0 row is "admission weight".
# - -1 means unknown -> NaN. Heights outside 100-250 cm and weights outside
#   20-300 kg are recording errors -> NaN.
# - Result: Height unknown for 5,754 stays (47.9%), admission weight for 1,003 (8.4%),
#   gender for 12, age and ICU type never missing. ICU types: medical 4,293, surgical 3,408,
#   cardiac surgery 2,530, coronary 1,769. Age 15 to 90 (median 67).
desc, ts = F.split_descriptors(long)
print("Descriptor table:", desc.shape)
missing = desc.isna().sum().to_frame("missing")
missing["missing_%"] = (missing["missing"] / len(desc) * 100).round(1)
display(missing)
display(desc["ICUType"].map(F.ICU_TYPES).value_counts().to_frame("stays"))
display(desc[["Age", "Height", "Weight"]].describe().round(1))

Descriptor table: (12000, 5)


,missing,missing_%
Parameter,,
Age,0,0.0
Gender,12,0.1
Height,5754,47.9
ICUType,0,0.0
Weight,1003,8.4


,stays
ICUType,
medical,4293
surgical,3408
cardiac_surgery,2530
coronary,1769


Parameter,Age,Height,Weight
count,12000.0,6246.0,10997.0
mean,64.5,169.5,81.2
std,17.2,10.9,23.5
min,15.0,121.9,21.7
25%,53.0,162.6,65.7
50%,67.0,170.2,78.0
75%,78.0,177.8,92.0
max,90.0,210.8,300.0


In [4]:
# ── Cell 4: clean the time series ───────────────────────────────────
# INSIGHTS
# - Rules live in src/features.py (VALID_RANGES) so every notebook uses the same ones.
# - Conservative on purpose: only physically impossible values go (e.g. Temp -17.8 C,
#   blood pressure 0, pH 0.4). Extreme but possible values stay, they carry the signal.
# - pH typed without the decimal point (735) is rescaled to 7.35 instead of dropped.
# - Result: 8,335 of 5,213,818 time-series rows removed (0.16%). Biggest rules:
#   DiasABP 1,939 and SysABP 1,891 (invasive BP of 0), NISysABP 552, NIDiasABP 474,
#   Weight 352, Temp 298. pH: 3 values rescaled, 18 removed. Also 2,403 rows in set-c
#   had a blank parameter name and were dropped.
n_raw = len(ts)
ts_clean, report = F.clean_time_series(ts)
per_param = ts["Parameter"].value_counts()
report["removed_%_of_param"] = [
    round(r.removed / per_param.get(r.parameter, np.nan) * 100, 2)
    for r in report.itertuples()
]
display(report)
print(f"Time-series rows: {n_raw:,} -> {len(ts_clean):,} "
      f"(removed {n_raw - len(ts_clean):,}, {(n_raw - len(ts_clean)) / n_raw:.2%})")
print("Variables left:", ts_clean["Parameter"].nunique())

,rule,parameter,removed,rescaled,removed_%_of_param
0,blank parameter name (cannot tell what was mea...,none,2403,0,NaN
1,minutes > 2880 (after 48 h),all,0,0,NaN
2,"pH rescaled (/10 or /100), not removed",pH,0,3,0.00
3,"outside (0, 300]",HR,45,0,0.01
4,"outside (25, 45]",Temp,298,0,0.11
5,"outside (0, 100]",RespRate,153,0,0.09
6,"outside (0, 300]",SysABP,1891,0,0.43
7,"outside (0, 300]",DiasABP,1939,0,0.44
8,"outside (0, 300]",MAP,86,0,0.02
9,"outside (0, 300]",NISysABP,552,0,0.19


Time-series rows: 5,213,818 -> 5,205,483 (removed 8,335, 0.16%)
Variables left: 37


In [5]:
# ── Cell 5: build one row per stay ──────────────────────────────────
# INSIGHTS
# - Per variable: min, max, mean, last value, count, and has_<var> (measured at all).
#   min/max catch the worst moment, mean the overall level, last the state at 48 h,
#   count how closely the patient was watched.
# - MechVent is only ever recorded as 1, so it becomes MechVent_ever (0/1).
# - ICUType becomes 4 one-hot columns. No imputation: unmeasured stays NaN.
# - Result: 12,000 rows x 225 features: 36 variables x 6 (min, max, mean, last,
#   count, has_) = 216, plus 9 static columns. 13 stays (all survivors) have no
#   time-series data at all, only descriptors. Kept for now, decided in modeling.
feats = F.build_features(ts_clean, desc)
n_series = len([c for c in feats.columns if c.startswith("has_")])
print(f"Feature table: {feats.shape[0]:,} rows x {feats.shape[1]} columns")
print(f"Time-series variables: {n_series} (x 6 = {n_series * 6} columns) + "
      f"{feats.shape[1] - n_series * 6} static columns")
display(feats.iloc[:3, :16])

Feature table: 12,000 rows x 225 columns
Time-series variables: 36 (x 6 = 216 columns) + 9 static columns


,Age,Gender,Height,Weight_admit,MechVent_ever,ICUType_coronary,ICUType_cardiac_surgery,ICUType_medical,ICUType_surgical,ALP_min,ALP_max,ALP_mean,ALP_last,ALP_count,ALT_min,ALT_max
RecordID,,,,,,,,,,,,,,,,
132539,54.0,0.0,NaN,NaN,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN
132540,76.0,1.0,175.3,76.0,1,0,1,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN
132541,44.0,0.0,NaN,56.7,1,0,0,1,0,105.0,127.0,116.0,105.0,2,75.0,91.0


In [6]:
# ── Cell 6: add label, baseline scores and split, then save ─────────
# INSIGHTS
# - The label and SAPS-I/SOFA come from the outcomes files. SAPS-I and SOFA are kept
#   only to compare against later, never as features (see NON_FEATURES).
# - Length_of_stay and Survival are dropped right here: they are known only after
#   the stay ends, so they would leak the outcome.
# - split/fold come from notebook 01's splits.csv, so every notebook uses the same folds.
# - Result: saved features.parquet: 12,000 rows, 225 features + 7 id/label/split columns.
outcomes = pd.concat([read_outcomes(RAW, s) for s in SETS], ignore_index=True)
outcomes = outcomes[["RecordID", LABEL, "SAPS-I", "SOFA"]]
outcomes[["SAPS-I", "SOFA"]] = outcomes[["SAPS-I", "SOFA"]].replace(-1, np.nan)
splits = pd.read_csv(PROCESSED / "splits.csv")

table = (feats.reset_index()
         .merge(splits, on="RecordID", how="left", validate="one_to_one")
         .merge(outcomes, on="RecordID", how="left", validate="one_to_one"))

NON_FEATURES = ["RecordID", "set", "split", "fold", LABEL, "SAPS-I", "SOFA"]
FEATURES = [c for c in table.columns if c not in NON_FEATURES]

out = PROCESSED / "features.parquet"
table.to_parquet(out, index=False)
print(f"Saved {out.relative_to(ROOT)}: {table.shape[0]:,} rows, {len(FEATURES)} features "
      f"+ {len(NON_FEATURES)} id/label/split columns")

Saved data\processed\features.parquet: 12,000 rows, 225 features + 7 id/label/split columns


In [7]:
# ── Cell 7: sanity checks ───────────────────────────────────────────
# INSIGHTS
# - One row per stay, every stay has a label and a split, no leakage columns.
# - Missing rate per variable = share of stays where it was never measured in 48 h.
# - Result: all checks pass. Mortality dev 14.03%, test 14.62% (same as notebook 01).
#   10 of 36 variables are missing in more than half of stays (TroponinI 95.3%,
#   Cholesterol 92.1%, TroponinT 78.0%, RespRate 72.3%). Even HR/BUN/Creatinine are
#   missing in about 1.5% of stays, so no variable is measured in every stay.
assert len(table) == 12_000
assert table["RecordID"].is_unique
assert table[LABEL].notna().all() and table["split"].notna().all()
assert not {"Length_of_stay", "Survival"} & set(table.columns)
assert not {"SAPS-I", "SOFA"} & set(FEATURES)
print("All checks passed.")
print("Mortality by split:", table.groupby("split")[LABEL].mean().round(4).to_dict())

has_cols = [c for c in FEATURES if c.startswith("has_")]
miss = (1 - table[has_cols].mean()).rename(lambda c: c[4:]).sort_values(ascending=False)
print(f"\nVariables never measured in more than half of stays: {(miss > 0.5).sum()} of {len(miss)}")
display((miss.head(15) * 100).round(1).to_frame("% of stays never measured"))
print("Measured in every stay:", list(miss[miss == 0].index))

All checks passed.


Mortality by split: {'dev': 0.1403, 'test': 0.1462}

Variables never measured in more than half of stays: 10 of 36


,% of stays never measured
TroponinI,95.3
Cholesterol,92.1
TroponinT,78.0
RespRate,72.3
Albumin,59.4
ALP,57.5
ALT,56.6
AST,56.6
Bilirubin,56.6
SaO2,55.3


Measured in every stay: []


In [8]:
# ── Cell 8: is missingness informative? ─────────────────────────────
# INSIGHTS
# - Doctors order a test when they are worried, so "was it measured" can predict death
#   on its own. If so, has_<var> flags and counts are real signal, and imputing a value
#   silently would throw that away.
# - Compared on the dev set (a+b) only, so the test set stays untouched.
# - Result: missingness is informative. Lactate was measured in 71.0% of deaths vs
#   52.4% of survivors; mortality is 18.1% when measured vs 9.0% when not. Same pattern
#   for pH (16.1% vs 7.6%), Albumin, ALT, Bilirubin, Troponins. Cholesterol is the
#   exception (13.1% vs 14.1%). More HR readings also means higher mortality:
#   10.4% in the lowest quartile vs 17.5% in the highest.
dev = table[table["split"] == "dev"]
rows = []
for var in ["Lactate", "TroponinT", "TroponinI", "Albumin", "ALT", "Bilirubin", "Cholesterol", "pH"]:
    measured = dev[f"has_{var}"] == 1
    rows.append({
        "variable": var,
        "measured_%_survivors": dev.loc[dev[LABEL] == 0, f"has_{var}"].mean() * 100,
        "measured_%_deaths": dev.loc[dev[LABEL] == 1, f"has_{var}"].mean() * 100,
        "mortality_%_if_measured": dev.loc[measured, LABEL].mean() * 100,
        "mortality_%_if_not": dev.loc[~measured, LABEL].mean() * 100,
    })
display(pd.DataFrame(rows).round(1))

q = pd.qcut(dev["HR_count"], 4, duplicates="drop")
display(dev.groupby(q, observed=True)[LABEL].agg(stays="size", mortality="mean")
        .assign(mortality=lambda d: (d["mortality"] * 100).round(1))
        .rename_axis("HR measurements in 48 h (quartile)"))

,variable,measured_%_survivors,measured_%_deaths,mortality_%_if_measured,mortality_%_if_not
0,Lactate,52.4,71.0,18.1,9.0
1,TroponinT,20.5,35.2,21.9,11.7
2,TroponinI,4.1,9.1,26.4,13.4
3,Albumin,38.2,55.3,19.1,10.6
4,ALT,40.7,58.7,19.1,10.2
5,Bilirubin,40.5,58.7,19.1,10.2
6,Cholesterol,8.2,7.6,13.1,14.1
7,pH,74.0,86.9,16.1,7.6


,stays,mortality
HR measurements in 48 h (quartile),,
"(-0.001, 48.0]",2017,10.4
"(48.0, 55.0]",2080,14.2
"(55.0, 65.0]",1989,14.3
"(65.0, 189.0]",1914,17.5


## Summary

- **Raw data:** 5.29 million measurements across 12,000 stays, all within the first 48 h.
- **Descriptors:** height is unknown for 48% of stays and admission weight for 8%. Age and ICU type are always present. Medical ICU is the largest group (4,293 stays).
- **Cleaning:** 8,335 impossible values removed (0.16% of rows), mostly invasive blood pressure readings of 0. 3 pH values rescaled (735 to 7.35). 2,403 set-c rows with a blank parameter name dropped.
- **Features:** 225 per stay. 36 variables x (min, max, mean, last, count, has_) + age, gender, height, admission weight, ever ventilated and 4 ICU-type columns. No imputation yet.
- **Missingness is informative:** stays where lactate was measured have twice the mortality (18.1% vs 9.0%), and patients who died had more HR readings. The `has_` flags and counts are real signal, not noise.
- **Watch for in modeling:** 13 stays have no time-series data at all, and 10 variables are missing in over half of stays, so models must handle NaN (gradient boosting does natively; logistic regression needs imputation + missingness flags inside the CV pipeline).

**Next:** `03_eda.ipynb` explores distributions, missingness patterns and mortality by age and ICU type on the dev set.